In [42]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
     

import torchvision
import torchvision.transforms as transforms

import sys
sys.path.insert(0, os.path.abspath(".."))

from models import *

global best_prec
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Using device:", device)
print('=> Building model...')

batch_size = 128

model_name = "resnet20_quant"
model = resnet20_quant()

# print(model)

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input, target = input.to(device), target.to(device)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.to(device), target.to(device)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [80, 120]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

Using device: mps
=> Building model...


In [ ]:
# This cell won't be given, but students will complete the training

lr = 0.1
weight_decay = 1e-4
epochs = 150
target_acc = 86.0  # stop training once test accuracy reaches this
best_prec = 0

#model = nn.DataParallel(model).cuda()
model.to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

    if prec >= target_acc:
        print('Reached target accuracy {:.1f}% at epoch {}, stopping training.'.format(target_acc, epoch))
        break

In [43]:
fdir = 'result/'+str(model_name)
PATH = fdir+'/model_best.pth.tar'
checkpoint = torch.load(PATH, map_location=device)
model.load_state_dict(checkpoint['state_dict'])

model.to(device)
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to device
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 8631/10000 (86%)



In [44]:
#send an input and grap the value by using prehook like HW3

class SaveOutput:
    def __init__(self):
        self.outputs = []
    def __call__(self, module, module_in):
        self.outputs.append(module_in)   # module_in is a tuple of the layer's inputs
    def clear(self):
        self.outputs = []

save_output = SaveOutput()
layer = model.layer1[0].conv1            # 2nd conv layer (1st quantized conv) of ResNet20
hook = layer.register_forward_pre_hook(save_output)   # grabs the layer input  (x)

layer_out = []                                        # grabs the layer output (original psum)
out_hook = layer.register_forward_hook(lambda module, module_in, module_out: layer_out.append(module_out))

dataiter = iter(testloader)
images, labels = next(dataiter)
images = images.to(device)

model.eval()
with torch.no_grad():
    out = model(images)
hook.remove()
out_hook.remove()

In [45]:
w_bit = 4
weight_q = layer.weight_q                # quantized value is stored during the training
w_alpha = layer.weight_quant.wgt_alpha   # learned clipping threshold for weights
w_delta = w_alpha / (2**(w_bit-1) - 1)   # 1 bit is the sign -> 3-bit magnitude -> 7 levels per side
weight_int = weight_q / w_delta          # integers in [-7, 7]
w_int = weight_int.detach().round().int().cpu()   # clean integer copy for display
print(f"w_int shape {tuple(w_int.shape)}  [Cout, Cin, Kh, Kw]")
print(f"w_alpha = {w_alpha.item():.4f}   w_delta = {w_delta.item():.4f}   range [{w_int.min()}, {w_int.max()}]")
print("\nfilter for output channel 0 (16 input channels, 3x3 each):")
for cin in range(w_int.shape[1]):
    rows = [" ".join(f"{v:3d}" for v in w_int[0, cin, r].tolist()) for r in range(3)]
    print(f"  cin {cin:2d}:  [{rows[0]}]  [{rows[1]}]  [{rows[2]}]")

w_int shape (16, 16, 3, 3)  [Cout, Cin, Kh, Kw]
w_alpha = 2.9078   w_delta = 0.4154   range [-7, 7]

filter for output channel 0 (16 input channels, 3x3 each):
  cin  0:  [ -6   1   7]  [ -5   2   5]  [ -7   0   6]
  cin  1:  [  4   3   1]  [ -2  -6  -6]  [  0  -2  -3]
  cin  2:  [  1   1   5]  [ -5  -5  -5]  [  1   5   6]
  cin  3:  [  1   2   1]  [  1  -3  -5]  [  1   2   1]
  cin  4:  [  0   1  -1]  [  0  -1  -1]  [  1   1   1]
  cin  5:  [  3  -3  -4]  [  0  -3   0]  [ -1  -4   0]
  cin  6:  [  7  -6  -7]  [ -1   7   5]  [  2  -7  -7]
  cin  7:  [ -2   1   0]  [ -2   0  -3]  [ -1   1   0]
  cin  8:  [  1   0   0]  [  0   1   1]  [  0   0   0]
  cin  9:  [ -6  -2   6]  [ -6   1   7]  [ -6  -1   3]
  cin 10:  [ -7   2   0]  [  7  -7   3]  [ -4   4  -1]
  cin 11:  [  3   5   7]  [ -4  -4  -2]  [ -1   1   1]
  cin 12:  [  5   2  -2]  [  2  -3  -7]  [  1   3   0]
  cin 13:  [  0   1   1]  [  7   3  -3]  [  5  -2  -6]
  cin 14:  [  0   4   2]  [ -5  -2   0]  [ -1  -1  -1]
  cin 15:  [  0

In [46]:
x_bit = 4
x = save_output.outputs[0][0]            # input of the 2nd conv layer
x_alpha = layer.act_alpha                # learned clipping threshold for activations
x_delta = x_alpha / (2**x_bit - 1)       # post-ReLU, unsigned -> 15 levels

act_quant_fn = act_quantization(x_bit) # define the quantization function
x_q = act_quant_fn(x, x_alpha)         # create the quantized value for x

x_int = x_q / x_delta                    # integers in [0, 15]
x_int_disp = x_int.detach().round().int().cpu()   # clean integer copy for display

# Adding pretty prints for x_int
print(f"x_int shape {tuple(x_int_disp.shape)}  [N, Cin, H, W]")
print(f"x_alpha = {x_alpha.item():.4f}   x_delta = {x_delta.item():.4f}   range [{x_int_disp.min()}, {x_int_disp.max()}]")
print("max |x_int - round|:", (x_int - x_int.round()).abs().max().item())
print("\nimage 0, input channel 0, top-left 8x16 pixels:")
for r in range(8):
    print("  " + " ".join(f"{v:2d}" for v in x_int_disp[0, 0, r, :16].tolist()))

x_int shape (128, 16, 32, 32)  [N, Cin, H, W]
x_alpha = 4.0089   x_delta = 0.2673   range [0, 15]
max |x_int - round|: 9.5367431640625e-07

image 0, input channel 0, top-left 8x16 pixels:
   4  4  4  4  4  4  4  4  5  5  4  3  4  5  5  5
   5  5  4  4  4  4  5  4  4  5  4  3  4  5  4  5
   5  5  4  3  4  5  5  4  5  5  3  3  4  5  4  4
   5  5  4  3  4  4  4  4  4  5  4  3  4  4  4  3
   4  4  4  3  4  4  5  4  4  5  7  4  0  3  3  1
   4  3  4  3  4  5  5  4  5  5  5  3  2  2  2  2
   3  1  3  7  6  5  5  4  4  5  4  3  3  3  3  4
   3  2  3  6  7  5  5  4  3  4  3  3  4  3  2  3


In [47]:
# psum from integers only: 4-bit x_int (0..15) * 4-bit w_int (-7..7), accumulated
conv_int = torch.nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, stride=1, padding=1, bias=False)
conv_int.weight = torch.nn.parameter.Parameter(weight_int)

with torch.no_grad():
    output_int = conv_int(x_int)                          # integer psum
    output_recovered = output_int * x_delta * w_delta     # rescale back to real units

#### original output of the layer, captured with the forward hook in cell 3

output_ref = layer_out[0]

#### difference between calculated (integer psum, rescaled) and hooked (original) results

difference = abs(output_ref - output_recovered)
print('mean |diff|           :', difference.mean().item())    # ~1e-5: float32 rounding only
print('max  |diff|           :', difference.max().item())
print('mean |hooked output|  :', output_ref.abs().mean().item())
print('relative error        :', (difference.norm() / output_ref.norm()).item())

mean |diff|           : 1.7643905039221863e-06
max  |diff|           : 2.288818359375e-05
mean |hooked output|  : 16.08553123474121
relative error        : 1.2069394017544255e-07
